# 17b · Fabric: dall'Hub direttamente alla Eventstream
*Blocco 5 · Dati in Fabric. Passo di creazione, dopo il 17a.*

Il percorso: **device → IoT Hub → route → endpoint di tipo *Fabric Eventstream* → Eventstream (sorgente *Custom endpoint*) → Eventhouse**.
L'Hub scrive con la **sua identità** (Microsoft Entra ID): nessuna chiave. I messaggi arrivano nello schema **CloudEvents**.

**Prima:** i notebook 01 e 02a-02c, e il **17a**, che crea in Fabric la Eventstream (sorgente *Custom endpoint*, destinazione *Eventhouse*) e passa qui i suoi valori.
**Qui:** si collega ogni Hub alla Eventstream: un endpoint e una route.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## I valori da collegare
Sono cinque. Arrivano dal **17a** (file `artifacts\17a-fabric-crea-eventstream\fabric.json`, letto da `_start.ps1`); se quel file non c'è, valgono quelli di `notebooks\_scenario.psd1`, sezione 6 (*Fabric*), compilati a mano.

| Valore | Dove si legge |
|---|---|
| `WorkspaceId` | l'indirizzo della Eventstream in Fabric, dopo `/groups/` |
| `EventstreamId` | lo stesso indirizzo, dopo `/eventstreams/` |
| `SourceId` | la sorgente *Custom endpoint*: la topologia della Eventstream (REST) |
| `EndpointUri` | i dettagli della sorgente, scheda *Entra ID Authentication*: *Event hub namespace*, con `sb://` davanti |
| `EntityPath` | gli stessi dettagli: *Event hub name* |

In [ ]:
$cfg.Fabric.GetEnumerator() | Sort-Object Name | ForEach-Object { [pscustomobject]@{ Valore = $_.Name; Contenuto = $_.Value } } | Format-Table -AutoSize | Out-String
$vuoti = @($cfg.Fabric.GetEnumerator() | Where-Object { -not $_.Value -or "$($_.Value)" -match '^<' })
if ($vuoti) { throw "Da popolare in _scenario.psd1 (sezione 6): $(($vuoti.Name) -join ', ')" }
"Tutti i valori ci sono." 


Valore        Contenuto
------        ---------
EndpointUri   sb://esehitn7qfhcxs1jsgm2hcx.servicebus.windows.net
EntityPath    esehitn7qfhcxs1jsgm2hcx_eh
EventstreamId 66a61069-a881-493c-a453-7f0c5b92198b
SourceId      50af879f-f78b-466d-b1e8-e84ee510c9e4
WorkspaceId   1bb5538f-ea39-4a44-8499-bcdb7fe0b2ef



Tutti i valori ci sono.

## Un passo a mano in Fabric: le identità degli Hub
Ogni Hub ha un'identità di sistema. Va aggiunta al **workspace** Fabric (*Manage access*) con ruolo **Contributor** (o superiore): senza, l'endpoint non si crea (errore 401).

In [ ]:
foreach ($h in $hubs) {
    $principal = az iot hub show --name $h.Name --resource-group $rg --query identity.principalId -o tsv --only-show-errors
    "Fabric > Manage access > aggiungi '$($h.Name)' (principalId $principal) come Contributor"
}

Fabric > Manage access > aggiungi 'iot-iot26-dev-uks-001' (principalId 7d9a6a8a-a9ba-45f9-9f4c-2cb9cf80c860) come Contributor

Fabric > Manage access > aggiungi 'iot-iot26-dev-uks-002' (principalId 8ee39000-1aa9-4f54-b486-85620f96a27f) come Contributor

## L'endpoint
Su ogni Hub un endpoint di tipo *Fabric Eventstream*, con l'identità di sistema. Il ruolo in Fabric impiega qualche istante a propagarsi: la cella riprova da sola.

In [ ]:
foreach ($h in $hubs) {
    $esistenti = @(az iot hub show --name $h.Name --resource-group $rg --query 'properties.routing.endpoints.eventStreams[].name' -o tsv --only-show-errors)
    $verbo = if ($esistenti -contains 'ep-eventstream') { 'update' } else { 'create' }   # se c'e' gia', lo si porta sui valori di _scenario.psd1
    $ok = $false
    for ($i = 1; $i -le 10 -and -not $ok; $i++) {
        az iot hub message-endpoint $verbo fabric-eventstream --hub-name $h.Name --resource-group $rg --endpoint-name 'ep-eventstream' `
            --endpoint-uri $fabric.EndpointUri --entity-path $fabric.EntityPath --workspace-id $fabric.WorkspaceId `
            --eventstream-id $fabric.EventstreamId --source-id $fabric.SourceId --identity '[system]' -o none --only-show-errors 2>$null
        if ($LASTEXITCODE -eq 0) { $ok = $true } else { "$($h.Name): attendo la propagazione del ruolo ($i/10)..."; Start-Sleep -Seconds 15 }
    }
    if (-not $ok) { throw "Endpoint non creato per $($h.Name): l'identita' dell'Hub non e' ancora nel workspace Fabric." }
    "$($h.Name): endpoint ep-eventstream ora punta alla Eventstream $($fabric.EventstreamId) ($verbo)."
}

iot-iot26-dev-uks-001: endpoint ep-eventstream creato.

iot-iot26-dev-uks-002: endpoint ep-eventstream creato.

## La route
La route manda i messaggi dei device all'endpoint. Da quel momento, ogni messaggio che l'Hub accetta arriva anche in Fabric.

In [ ]:
foreach ($h in $hubs) {
    $routes = @(az iot hub message-route list --hub-name $h.Name --resource-group $rg --query '[].name' -o tsv --only-show-errors)
    if ($routes -notcontains 'route-eventstream') {
        az iot hub message-route create --hub-name $h.Name --resource-group $rg --route-name 'route-eventstream' --endpoint-name 'ep-eventstream' --source-type devicemessages -o none
        Assert-Az "Route verso la Eventstream ($($h.Name))"
    }
    "--- $($h.Name)"
    az iot hub message-route list --hub-name $h.Name --resource-group $rg --query '[].{route:name, endpoint:endpointNames[0], sorgente:source, attiva:isEnabled}' -o table
}

--- iot-iot26-dev-uks-001

Route              Endpoint        Sorgente        Attiva

-----------------  --------------  --------------  --------

route-eventstream  ep-eventstream  DeviceMessages  True

--- iot-iot26-dev-uks-002

Route              Endpoint        Sorgente        Attiva

-----------------  --------------  --------------  --------

route-eventstream  ep-eventstream  DeviceMessages  True

Prossimo passo: **18**, i device mandano i messaggi.